# 08. `MFE - downside` 직접 utility 학습 (`λ=1`)

두 번째 개선 항목만 단독 검증한다. `06_modern_tcn_baseline.ipynb`와 데이터, 입력 feature, context, 날짜 OOF fold, stride=3, scaler, Compact ModernTCN, optimizer와 epoch 후보를 동일하게 유지한다.

기존 방식은 MFE와 downside를 두 개의 head로 각각 예측한 뒤 뺐다. 두 예측이 같은 변동성 요인으로 붕괴하면서 derived utility Spearman이 낮았다. 이번에는 처음부터 아래 값 하나를 직접 학습한다.

```text
downside = -MAE_3m
excursion_utility = MFE_3m - 1.0 × downside
```

- λ는 추가 tuning 없이 `1.0`으로 고정한다.
- fold Train의 q1~q99로 clip하고 median/IQR로 scaling한다.
- loss는 `SmoothL1(beta=0.5)`다.
- OOF epoch 선택 기준은 실제 utility와 예측 utility의 Spearman이다.

이 utility는 3분 안의 두 극값 차이이며 체결 순서를 반영한 실현 수익률이 아니다. 아직 episode 평가, percentile threshold, rolling calibration, absolute price/time context는 적용하지 않는다.

In [1]:
from pathlib import Path
import json
import math
import random
import time
from copy import deepcopy

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    roc_auc_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
PREPROCESS_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model/moderntcn_direct_utility_lambda1_v1').resolve()
RESULT_ROOT = (PROJECT_ROOT / '../../results/training/moderntcn_direct_utility_lambda1_v1').resolve()
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_VERSION = 'ohlc_60m_tp3pct_v1'
RANDOM_VERSION = 'ohlc_60m_tp3pct_random_entry_v1'
SEED = 20260721
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONFIG = {
    'sequence_length': 60,
    'input_features': 11,
    'context_features': 20,
    'd_model': 16,
    'd_ff': 48,
    'num_blocks': 2,
    'patch_size': 8,
    'patch_stride': 4,
    'large_kernel_size': 7,
    'small_kernel_size': 3,
    'dropout': 0.15,
    'batch_size': 512,
    'candidate_epochs': [4, 8, 12],
    'learning_rate': 3e-4,
    'weight_decay': 1e-3,
    'gradient_clip': 1.0,
    'train_stride_minutes': 3,
    'input_clip': 10.0,
    'regression_quantile_clip': 0.99,
    'smooth_l1_beta': 0.5,
    'num_workers': 0,
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
torch.set_float32_matmul_precision('high')

print('project:', PROJECT_ROOT)
print('device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)
print('model output:', MODEL_ROOT)
print('result output:', RESULT_ROOT)

project: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge
device: cuda
GPU: NVIDIA RTX 6000 Ada Generation
torch: 2.9.1+cu128
model output: /home/user/urbandatalab/YSLee/model/moderntcn_direct_utility_lambda1_v1
result output: /home/user/urbandatalab/YSLee/results/training/moderntcn_direct_utility_lambda1_v1


## 1. 전처리 artifact와 날짜 split 로드

기존 hard TP는 참고 metric으로만 사용하고 학습 target은 `MFE_3m + MAE_3m` 하나다. scaling과 q1/q99 clipping은 OOF fold의 과거 fit 날짜에서만 계산한다.

In [2]:
schema = json.loads((PREPROCESS_ROOT / f'{DATA_VERSION}_schema.json').read_text(encoding='utf-8'))
fold_document = json.loads((PREPROCESS_ROOT / f'{DATA_VERSION}_walk_forward_folds.json').read_text(encoding='utf-8'))

metadata = pd.read_parquet(PREPROCESS_ROOT / f'{DATA_VERSION}_metadata.parquet')
random_labels = pd.read_parquet(PREPROCESS_ROOT / f'{RANDOM_VERSION}_metadata.parquet')
split = pd.read_parquet(PREPROCESS_ROOT / f'{DATA_VERSION}_split.parquet')

with np.load(PREPROCESS_ROOT / f'{DATA_VERSION}_features.npz') as arrays:
    sequence = arrays['sequence'].astype(np.float32, copy=False)
    context = arrays['context'].astype(np.float32, copy=False)

assert metadata['sample_id'].is_unique
assert metadata['sample_id'].tolist() == random_labels['sample_id'].tolist() == split['sample_id'].tolist()
assert len(metadata) == len(sequence) == len(context)
assert sequence.shape[1:] == (CONFIG['sequence_length'], CONFIG['input_features'])
assert context.shape[1] == CONFIG['context_features']
assert np.isfinite(sequence).all() and np.isfinite(context).all()
assert fold_document['test_is_pristine'] is False

y_hard = metadata['target_tp3_3m'].to_numpy(np.float32)
y_soft = random_labels['p_tp3_random_entry_3m'].to_numpy(np.float32)
mfe_3m = metadata['mfe_3m'].to_numpy(np.float32)
downside_3m = (-metadata['mae_3m']).to_numpy(np.float32)
LAMBDA_DOWNSIDE = 1.0
y_utility = (mfe_3m - LAMBDA_DOWNSIDE * downside_3m).astype(np.float32)

assert np.isin(y_hard, [0.0, 1.0]).all()
assert np.isfinite(y_utility).all()

# 기존 실험과 동일한 stride=3 학습 sampling. OOF/Test 평가는 모든 시점에서 한다.
within_run_position = metadata.groupby('run_id', sort=False).cumcount().to_numpy()
train_sampling_mask = (within_run_position % CONFIG['train_stride_minutes']) == 0

train_mask = split['split'].eq('train').to_numpy()
test_mask = split['split'].eq('test').to_numpy()
oof_mask = split['oof_fold'].gt(0).to_numpy()

dataset_summary = pd.DataFrame({
    'partition': ['Train 전체', 'Train 학습 stride=3', 'OOF 평가', 'Test 평가'],
    'samples': [
        int(train_mask.sum()),
        int((train_mask & train_sampling_mask).sum()),
        int(oof_mask.sum()),
        int(test_mask.sum()),
    ],
})
display(dataset_summary)

utility_summary = pd.DataFrame([
    {
        'partition': name,
        'mean': float(y_utility[mask].mean()),
        'q01': float(np.quantile(y_utility[mask], 0.01)),
        'q05': float(np.quantile(y_utility[mask], 0.05)),
        'median': float(np.median(y_utility[mask])),
        'q95': float(np.quantile(y_utility[mask], 0.95)),
        'q99': float(np.quantile(y_utility[mask], 0.99)),
        'positive_share': float((y_utility[mask] > 0).mean()),
        'tp3_rate': float(y_hard[mask].mean()),
    }
    for name, mask in [('Train', train_mask), ('OOF', oof_mask), ('Test', test_mask)]
])
display(utility_summary.style.format({
    'mean': '{:.4%}', 'q01': '{:.4%}', 'q05': '{:.4%}',
    'median': '{:.4%}', 'q95': '{:.4%}', 'q99': '{:.4%}',
    'positive_share': '{:.2%}', 'tp3_rate': '{:.2%}',
}))
print('sequence:', sequence.shape, 'context:', context.shape)
print('OOF folds:', len(fold_document['folds']))

,partition,samples
0,Train 전체,53047
1,Train 학습 stride=3,17756
2,OOF 평가,31081
3,Test 평가,11097


,partition,mean,q01,q05,median,q95,q99,positive_share,tp3_rate
0,Train,0.0290%,-8.0605%,-4.0549%,0.0000%,4.1870%,8.8555%,47.14%,12.37%
1,OOF,0.0522%,-8.6668%,-4.3091%,0.0000%,4.5643%,9.7034%,47.55%,13.82%
2,Test,0.0903%,-6.2513%,-3.1259%,0.0000%,3.5093%,6.9108%,47.25%,9.53%


sequence: (64144, 60, 11) context: (64144, 20)
OOF folds: 4


## 2. 고정된 Compact ModernTCN 백본

06·07번과 동일한 117,825 parameter 모델을 사용한다. output만 하나의 utility regression 값으로 해석한다.

In [3]:
class ConvBN(nn.Module):
    def __init__(self, channels, kernel_size):
        super().__init__()
        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=channels,
            bias=False,
        )
        self.bn = nn.BatchNorm1d(channels)

    def forward(self, x):
        return self.bn(self.conv(x))


class ReparamLargeSmallKernel(nn.Module):
    def __init__(self, channels, large_kernel, small_kernel):
        super().__init__()
        self.large_branch = ConvBN(channels, large_kernel)
        self.small_branch = ConvBN(channels, small_kernel)

    def forward(self, x):
        return self.large_branch(x) + self.small_branch(x)


class ModernTCNBlock(nn.Module):
    def __init__(
        self,
        n_variables,
        d_model,
        d_ff,
        large_kernel,
        small_kernel,
        dropout,
    ):
        super().__init__()
        channels = n_variables * d_model
        self.n_variables = n_variables
        self.d_model = d_model
        self.d_ff = d_ff

        self.temporal_mixer = ReparamLargeSmallKernel(
            channels, large_kernel, small_kernel
        )
        self.feature_norm = nn.BatchNorm1d(d_model)

        # 각 변수 내부 embedding을 독립적으로 변환한다.
        self.within_variable_ffn = nn.Sequential(
            nn.Conv1d(channels, n_variables * d_ff, kernel_size=1, groups=n_variables),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(n_variables * d_ff, channels, kernel_size=1, groups=n_variables),
            nn.Dropout(dropout),
        )

        # embedding channel별로 변수 축을 섞는다.
        self.cross_variable_ffn = nn.Sequential(
            nn.Conv1d(channels, n_variables * d_ff, kernel_size=1, groups=d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(n_variables * d_ff, channels, kernel_size=1, groups=d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: [B, M, D, N]
        residual = x
        batch, variables, embedding, patches = x.shape

        x = self.temporal_mixer(x.reshape(batch, variables * embedding, patches))
        x = x.reshape(batch * variables, embedding, patches)
        x = self.feature_norm(x)
        x = x.reshape(batch, variables * embedding, patches)

        x = self.within_variable_ffn(x)
        x = x.reshape(batch, variables, embedding, patches)

        x = x.permute(0, 2, 1, 3).reshape(batch, embedding * variables, patches)
        x = self.cross_variable_ffn(x)
        x = x.reshape(batch, embedding, variables, patches).permute(0, 2, 1, 3)
        return residual + x


class CompactModernTCN(nn.Module):
    def __init__(
        self,
        input_features,
        context_features,
        output_dim,
        d_model=16,
        d_ff=48,
        num_blocks=2,
        patch_size=8,
        patch_stride=4,
        large_kernel_size=7,
        small_kernel_size=3,
        dropout=0.15,
    ):
        super().__init__()
        self.input_features = input_features
        self.patch_size = patch_size
        self.patch_stride = patch_stride
        self.patch_embedding = nn.Sequential(
            nn.Conv1d(1, d_model, kernel_size=patch_size, stride=patch_stride),
            nn.BatchNorm1d(d_model),
        )
        self.blocks = nn.ModuleList([
            ModernTCNBlock(
                input_features,
                d_model,
                d_ff,
                large_kernel_size,
                small_kernel_size,
                dropout,
            )
            for _ in range(num_blocks)
        ])
        self.output_norm = nn.BatchNorm1d(d_model)
        self.context_encoder = nn.Sequential(
            nn.Linear(context_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        pooled_dim = input_features * d_model * 3 + d_model
        self.head = nn.Sequential(
            nn.Linear(pooled_dim, d_ff * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff * 2, output_dim),
        )
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Conv1d)):
            if module.weight is not None:
                nn.init.kaiming_normal_(module.weight, nonlinearity='linear')
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, sequence_x, context_x):
        # [B, T, M] -> 변수별 patch embedding -> [B, M, D, N]
        x = sequence_x.transpose(1, 2)
        batch, variables, length = x.shape
        pad_length = self.patch_size - self.patch_stride
        if pad_length > 0:
            x = torch.cat([x, x[:, :, -1:].expand(-1, -1, pad_length)], dim=-1)
        x = self.patch_embedding(x.reshape(batch * variables, 1, -1))
        embedding, patches = x.shape[1], x.shape[2]
        x = x.reshape(batch, variables, embedding, patches)

        for block in self.blocks:
            x = block(x)

        x = self.output_norm(x.reshape(batch * variables, embedding, patches))
        x = x.reshape(batch, variables, embedding, patches)
        pooled = torch.cat([
            x.mean(dim=-1),
            x.amax(dim=-1),
            x[:, :, :, -1],
        ], dim=2).reshape(batch, -1)
        fused = torch.cat([pooled, self.context_encoder(context_x)], dim=1)
        return self.head(fused)


smoke_model = CompactModernTCN(
    input_features=CONFIG['input_features'],
    context_features=CONFIG['context_features'],
    output_dim=1,
    d_model=CONFIG['d_model'],
    d_ff=CONFIG['d_ff'],
    num_blocks=CONFIG['num_blocks'],
    patch_size=CONFIG['patch_size'],
    patch_stride=CONFIG['patch_stride'],
    large_kernel_size=CONFIG['large_kernel_size'],
    small_kernel_size=CONFIG['small_kernel_size'],
    dropout=CONFIG['dropout'],
).to(DEVICE)
with torch.inference_mode():
    smoke_output = smoke_model(
        torch.zeros(2, 60, 11, device=DEVICE),
        torch.zeros(2, 20, device=DEVICE),
    )
MODEL_PARAMETER_COUNT = sum(p.numel() for p in smoke_model.parameters())
print('parameters:', f'{MODEL_PARAMETER_COUNT:,}')
print('TimeMixer++ adaptation parameters: 130,999')
print('parameter ratio:', f'{MODEL_PARAMETER_COUNT / 130999:.3f}')
print('smoke output:', tuple(smoke_output.shape))
del smoke_model, smoke_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()

parameters: 117,825
TimeMixer++ adaptation parameters: 130,999
parameter ratio: 0.899
smoke output: (2, 1)


## 3. Fold-local input/utility scaling과 SmoothL1

- 입력 median/IQR과 utility q1/q99·median/IQR은 각 fold의 fit 날짜에서만 계산한다.
- utility의 양·음 극단값은 q1/q99로 clip한다.
- Test는 scaling, λ, epoch 선택에 사용하지 않는다.

In [4]:
STRATEGIES = {
    'direct_utility': {'output_dim': 1, 'selection_metric': 'utility_spearman', 'mode': 'max'},
}


def robust_center_scale(values, axis=0):
    center = np.median(values, axis=axis).astype(np.float32)
    q25 = np.quantile(values, 0.25, axis=axis).astype(np.float32)
    q75 = np.quantile(values, 0.75, axis=axis).astype(np.float32)
    scale = (q75 - q25).astype(np.float32)
    scale = np.where(scale < 1e-6, 1.0, scale).astype(np.float32)
    return center, scale


def fit_input_scaler(indices):
    seq_values = sequence[indices].reshape(-1, sequence.shape[-1])
    seq_center, seq_scale = robust_center_scale(seq_values, axis=0)
    ctx_center, ctx_scale = robust_center_scale(context[indices], axis=0)
    return {
        'sequence_center': seq_center,
        'sequence_scale': seq_scale,
        'context_center': ctx_center,
        'context_scale': ctx_scale,
    }


def transform_inputs(indices, scaler):
    seq = (sequence[indices] - scaler['sequence_center'][None, None, :]) / scaler['sequence_scale'][None, None, :]
    ctx = (context[indices] - scaler['context_center'][None, :]) / scaler['context_scale'][None, :]
    seq = np.clip(seq, -CONFIG['input_clip'], CONFIG['input_clip']).astype(np.float32)
    ctx = np.clip(ctx, -CONFIG['input_clip'], CONFIG['input_clip']).astype(np.float32)
    return np.ascontiguousarray(seq), np.ascontiguousarray(ctx)


def fit_utility_scaler(indices):
    values = y_utility[indices]
    lower, upper = np.quantile(values, [0.01, 0.99]).astype(np.float32)
    clipped = np.clip(values, lower, upper)
    center, scale = robust_center_scale(clipped, axis=0)
    return {
        'lower': np.asarray(lower, dtype=np.float32),
        'upper': np.asarray(upper, dtype=np.float32),
        'center': np.asarray(center, dtype=np.float32),
        'scale': np.asarray(scale, dtype=np.float32),
    }


def transform_utility_target(indices, scaler):
    values = np.clip(y_utility[indices], scaler['lower'], scaler['upper'])
    scaled = (values - scaler['center']) / scaler['scale']
    return scaled.astype(np.float32)[:, None]


def inverse_utility_target(values, scaler):
    restored = values.reshape(-1) * scaler['scale'] + scaler['center']
    return np.clip(restored, scaler['lower'], scaler['upper']).astype(np.float32)


def make_loader(indices, input_scaler, target_scaler, shuffle=False, seed=SEED):
    seq, ctx = transform_inputs(indices, input_scaler)
    target = transform_utility_target(indices, target_scaler)
    dataset = TensorDataset(
        torch.from_numpy(seq),
        torch.from_numpy(ctx),
        torch.from_numpy(np.ascontiguousarray(target)),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=shuffle,
        num_workers=CONFIG['num_workers'],
        pin_memory=DEVICE.type == 'cuda',
        drop_last=False,
        generator=generator,
    )


def build_model():
    return CompactModernTCN(
        input_features=CONFIG['input_features'],
        context_features=CONFIG['context_features'],
        output_dim=1,
        d_model=CONFIG['d_model'],
        d_ff=CONFIG['d_ff'],
        num_blocks=CONFIG['num_blocks'],
        patch_size=CONFIG['patch_size'],
        patch_stride=CONFIG['patch_stride'],
        large_kernel_size=CONFIG['large_kernel_size'],
        small_kernel_size=CONFIG['small_kernel_size'],
        dropout=CONFIG['dropout'],
    ).to(DEVICE)


def predict(model, loader):
    model.eval()
    outputs = []
    with torch.inference_mode():
        for seq_batch, ctx_batch, _ in loader:
            seq_batch = seq_batch.to(DEVICE, non_blocking=True)
            ctx_batch = ctx_batch.to(DEVICE, non_blocking=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=DEVICE.type == 'cuda',
            ):
                output = model(seq_batch, ctx_batch)
            outputs.append(output.float().cpu().numpy())
    return np.concatenate(outputs)


def safe_spearman(actual, predicted):
    value = spearmanr(actual, predicted).statistic
    return float(value) if np.isfinite(value) else np.nan


def utility_metrics(indices, scaled_prediction, target_scaler):
    prediction = inverse_utility_target(scaled_prediction, target_scaler)
    actual = y_utility[indices]
    return {
        'utility_spearman': safe_spearman(actual, prediction),
        'utility_mae': float(mean_absolute_error(actual, prediction)),
        'utility_rmse': float(np.sqrt(np.mean((actual - prediction) ** 2))),
        'hard_tp3_pr_auc': float(average_precision_score(y_hard[indices], prediction)),
        'hard_tp3_roc_auc': float(roc_auc_score(y_hard[indices], prediction)),
        'prediction_mean': float(prediction.mean()),
        'target_mean': float(actual.mean()),
        'prediction_vs_mfe_spearman': safe_spearman(mfe_3m[indices], prediction),
        'prediction_vs_downside_spearman': safe_spearman(downside_3m[indices], prediction),
    }


def train_epochs(fit_indices, eval_indices, candidate_epochs, run_seed):
    seed_everything(run_seed)
    input_scaler = fit_input_scaler(fit_indices)
    target_scaler = fit_utility_scaler(fit_indices)
    train_loader = make_loader(
        fit_indices, input_scaler, target_scaler, shuffle=True, seed=run_seed
    )
    eval_loader = make_loader(
        eval_indices, input_scaler, target_scaler, shuffle=False, seed=run_seed
    )
    model = build_model()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(candidate_epochs), eta_min=CONFIG['learning_rate'] * 0.1
    )
    history, predictions = [], {}

    for epoch in range(1, max(candidate_epochs) + 1):
        model.train()
        total_loss = 0.0
        total_count = 0
        for seq_batch, ctx_batch, target_batch in train_loader:
            seq_batch = seq_batch.to(DEVICE, non_blocking=True)
            ctx_batch = ctx_batch.to(DEVICE, non_blocking=True)
            target_batch = target_batch.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=DEVICE.type == 'cuda',
            ):
                output = model(seq_batch, ctx_batch)
                loss = F.smooth_l1_loss(
                    output, target_batch, beta=CONFIG['smooth_l1_beta']
                )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip'])
            optimizer.step()
            total_loss += float(loss.detach()) * len(target_batch)
            total_count += len(target_batch)
        scheduler.step()
        history.append({
            'epoch': epoch,
            'train_loss': total_loss / total_count,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })
        if epoch in candidate_epochs:
            predictions[epoch] = predict(model, eval_loader)

    del train_loader, eval_loader, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        'predictions': predictions,
        'history': history,
        'input_scaler': input_scaler,
        'target_scaler': target_scaler,
    }

## 4. Direct utility expanding walk-forward OOF 학습

과거 날짜만 fit하고 다음 한 날짜의 utility를 예측한다. Test는 선택에 사용하지 않는다.

In [5]:
fold_records = []
history_records = []
oof_raw_predictions = {
    epoch: np.full((len(metadata), 1), np.nan, dtype=np.float32)
    for epoch in CONFIG['candidate_epochs']
}
oof_target_scalers = {}

oof_start = time.time()
print('===== OOF strategy: direct_utility =====')
for fold in fold_document['folds']:
    fold_number = int(fold['fold'])
    fit_all = np.flatnonzero(split['session'].isin(fold['fit_sessions']).to_numpy())
    fit_indices = fit_all[train_sampling_mask[fit_all]]
    eval_indices = np.flatnonzero(split['session'].eq(fold['evaluation_session']).to_numpy())
    assert split.iloc[fit_indices]['decision_timestamp'].max() < split.iloc[eval_indices]['decision_timestamp'].min()

    result = train_epochs(
        fit_indices, eval_indices, CONFIG['candidate_epochs'], SEED + fold_number
    )
    oof_target_scalers[fold_number] = result['target_scaler']
    for row in result['history']:
        history_records.append({
            'phase': 'oof', 'strategy': 'direct_utility', 'fold': fold_number,
            'fit_sessions': ','.join(fold['fit_sessions']),
            'evaluation_session': fold['evaluation_session'],
            'fit_samples_after_stride': len(fit_indices), **row,
        })
    for epoch, raw_prediction in result['predictions'].items():
        oof_raw_predictions[epoch][eval_indices] = raw_prediction
        fold_records.append({
            'strategy': 'direct_utility',
            'fold': fold_number,
            'evaluation_session': fold['evaluation_session'],
            'epoch': epoch,
            'fit_samples_after_stride': len(fit_indices),
            'evaluation_samples': len(eval_indices),
            **utility_metrics(eval_indices, raw_prediction, result['target_scaler']),
        })
    value = fold_records[-1]['utility_spearman']
    print(
        f"fold {fold_number} | fit {len(fit_indices):,} | eval {len(eval_indices):,} "
        f"| epoch {max(CONFIG['candidate_epochs'])} utility_spearman={value:.5f}"
    )

print(f'OOF elapsed: {(time.time() - oof_start) / 60:.2f} min')
fold_metrics = pd.DataFrame(fold_records)
training_history = pd.DataFrame(history_records)
display(fold_metrics.head())

===== OOF strategy: direct_utility =====


fold 1 | fit 7,355 | eval 9,077 | epoch 12 utility_spearman=0.09098


fold 2 | fit 10,397 | eval 7,297 | epoch 12 utility_spearman=0.09255


fold 3 | fit 12,838 | eval 6,120 | epoch 12 utility_spearman=0.06547


fold 4 | fit 14,883 | eval 8,587 | epoch 12 utility_spearman=0.08287
OOF elapsed: 0.25 min


,strategy,fold,evaluation_session,epoch,fit_samples_after_stride,evaluation_samples,utility_spearman,utility_mae,utility_rmse,hard_tp3_pr_auc,hard_tp3_roc_auc,prediction_mean,target_mean,prediction_vs_mfe_spearman,prediction_vs_downside_spearman
0,direct_utility,1,session_2026-07-10,4,7355,9077,0.090325,0.020813,0.035542,0.214064,0.467756,0.001207,0.000343,0.022527,-0.120102
1,direct_utility,1,session_2026-07-10,8,7355,9077,0.090505,0.020755,0.035538,0.184901,0.399978,-0.000442,0.000343,-0.088732,-0.242168
2,direct_utility,1,session_2026-07-10,12,7355,9077,0.090980,0.020760,0.035565,0.182818,0.393721,-0.000578,0.000343,-0.095500,-0.250811
3,direct_utility,2,session_2026-07-13,4,10397,7297,0.076831,0.018200,0.029632,0.126713,0.378504,-0.001342,0.001111,-0.106858,-0.238246
4,direct_utility,2,session_2026-07-13,8,10397,7297,0.091149,0.018135,0.029553,0.142775,0.422256,-0.000059,0.001111,-0.036131,-0.187811


## 5. OOF utility Spearman으로 epoch 선택

Fold마다 target scaling이 다르므로 원래 utility 단위로 복원한 뒤 네 OOF 날짜를 이어 붙여 평가한다.

In [6]:
epoch_records = []
oof_indices = np.flatnonzero(oof_mask)
oof_fold_numbers = split.iloc[oof_indices]['oof_fold'].to_numpy()

for epoch in CONFIG['candidate_epochs']:
    raw = oof_raw_predictions[epoch][oof_indices]
    assert np.isfinite(raw).all()
    restored = np.full(len(oof_indices), np.nan, dtype=np.float32)
    for fold in fold_document['folds']:
        fold_number = int(fold['fold'])
        mask = oof_fold_numbers == fold_number
        restored[mask] = inverse_utility_target(
            raw[mask], oof_target_scalers[fold_number]
        )
    actual = y_utility[oof_indices]
    daily = fold_metrics[fold_metrics['epoch'].eq(epoch)]
    epoch_records.append({
        'strategy': 'direct_utility',
        'epoch': epoch,
        'utility_spearman': safe_spearman(actual, restored),
        'utility_mae': float(mean_absolute_error(actual, restored)),
        'utility_rmse': float(np.sqrt(np.mean((actual - restored) ** 2))),
        'hard_tp3_pr_auc': float(average_precision_score(y_hard[oof_indices], restored)),
        'hard_tp3_roc_auc': float(roc_auc_score(y_hard[oof_indices], restored)),
        'prediction_vs_mfe_spearman': safe_spearman(mfe_3m[oof_indices], restored),
        'prediction_vs_downside_spearman': safe_spearman(downside_3m[oof_indices], restored),
        'worst_daily_utility_spearman': float(daily['utility_spearman'].min()),
    })

epoch_selection = pd.DataFrame(epoch_records)
selected_epoch = int(
    epoch_selection.loc[epoch_selection['utility_spearman'].idxmax(), 'epoch']
)
display(epoch_selection)
print('selected epoch:', selected_epoch)

,strategy,epoch,utility_spearman,utility_mae,utility_rmse,hard_tp3_pr_auc,hard_tp3_roc_auc,prediction_vs_mfe_spearman,prediction_vs_downside_spearman,worst_daily_utility_spearman
0,direct_utility,4,0.074663,0.018953,0.031857,0.172311,0.450658,-0.015188,-0.139691,0.063258
1,direct_utility,8,0.082916,0.018908,0.031858,0.159894,0.417162,-0.048928,-0.188310,0.063405
2,direct_utility,12,0.083609,0.018913,0.031882,0.160862,0.416938,-0.044200,-0.185328,0.065472


selected epoch: 12


## 6. 전체 Train 재학습과 고정 Test 진단

선택된 epoch로 전체 Train을 재학습하고 고정 Test utility를 한 번 예측한다.

In [7]:
def train_final_model(fit_indices, eval_indices, selected_epoch, run_seed):
    seed_everything(run_seed)
    input_scaler = fit_input_scaler(fit_indices)
    target_scaler = fit_utility_scaler(fit_indices)
    train_loader = make_loader(
        fit_indices, input_scaler, target_scaler, shuffle=True, seed=run_seed
    )
    eval_loader = make_loader(
        eval_indices, input_scaler, target_scaler, shuffle=False, seed=run_seed
    )
    model = build_model()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(CONFIG['candidate_epochs']),
        eta_min=CONFIG['learning_rate'] * 0.1,
    )
    history = []
    for epoch in range(1, selected_epoch + 1):
        model.train()
        total_loss = 0.0
        total_count = 0
        for seq_batch, ctx_batch, target_batch in train_loader:
            seq_batch = seq_batch.to(DEVICE, non_blocking=True)
            ctx_batch = ctx_batch.to(DEVICE, non_blocking=True)
            target_batch = target_batch.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=DEVICE.type == 'cuda',
            ):
                output = model(seq_batch, ctx_batch)
                loss = F.smooth_l1_loss(
                    output, target_batch, beta=CONFIG['smooth_l1_beta']
                )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip'])
            optimizer.step()
            total_loss += float(loss.detach()) * len(target_batch)
            total_count += len(target_batch)
        scheduler.step()
        history.append({
            'phase': 'final', 'strategy': 'direct_utility', 'fold': 0,
            'fit_sessions': ','.join(fold_document['train_sessions']),
            'evaluation_session': ','.join(fold_document['test_sessions']),
            'fit_samples_after_stride': len(fit_indices),
            'epoch': epoch, 'train_loss': total_loss / total_count,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })
    prediction = predict(model, eval_loader)
    return model, input_scaler, target_scaler, prediction, history


full_train_all = np.flatnonzero(train_mask)
full_train_indices = full_train_all[train_sampling_mask[full_train_all]]
test_indices = np.flatnonzero(test_mask)
final_start = time.time()
final_model, final_input_scaler, final_target_scaler, test_raw_prediction, final_history = train_final_model(
    full_train_indices, test_indices, selected_epoch, SEED + 1000
)
history_records.extend(final_history)
training_history = pd.DataFrame(history_records)

test_metric_records = [{
    'strategy': 'direct_utility', 'scope': 'all_test', 'session': 'ALL',
    'selected_epoch': selected_epoch, 'samples': len(test_indices),
    **utility_metrics(test_indices, test_raw_prediction, final_target_scaler),
}]
for session in fold_document['test_sessions']:
    local = np.flatnonzero(split['session'].eq(session).to_numpy())
    positions = np.searchsorted(test_indices, local)
    test_metric_records.append({
        'strategy': 'direct_utility', 'scope': 'daily', 'session': session,
        'selected_epoch': selected_epoch, 'samples': len(local),
        **utility_metrics(local, test_raw_prediction[positions], final_target_scaler),
    })
test_metrics = pd.DataFrame(test_metric_records)
print(f'final elapsed: {(time.time() - final_start) / 60:.2f} min')
display(test_metrics)

final elapsed: 0.09 min


,strategy,scope,session,selected_epoch,samples,utility_spearman,utility_mae,utility_rmse,hard_tp3_pr_auc,hard_tp3_roc_auc,prediction_mean,target_mean,prediction_vs_mfe_spearman,prediction_vs_downside_spearman
0,direct_utility,all_test,ALL,12,11097,0.085540,0.013883,0.022441,0.118091,0.436488,-0.000069,0.000903,-0.023573,-0.160263
1,direct_utility,daily,session_2026-07-16,12,5338,0.076226,0.015613,0.024898,0.134277,0.437610,-0.000171,0.001280,-0.033678,-0.165221
2,direct_utility,daily,session_2026-07-17,12,5759,0.094089,0.012279,0.019894,0.106385,0.438661,0.000026,0.000555,-0.010478,-0.150506


## 7. Target·모델·예측·지표 저장

기존 MFE·MAE 모델과 결과를 덮어쓰지 않고 λ=1 direct utility version으로 저장한다.

In [8]:
def scaler_to_serializable(scaler):
    return {key: float(np.asarray(value)) for key, value in scaler.items()}


def restore_oof_prediction(epoch):
    raw = oof_raw_predictions[epoch][oof_indices]
    restored = np.full(len(oof_indices), np.nan, dtype=np.float32)
    for fold in fold_document['folds']:
        fold_number = int(fold['fold'])
        mask = oof_fold_numbers == fold_number
        restored[mask] = inverse_utility_target(
            raw[mask], oof_target_scalers[fold_number]
        )
    return restored


oof_prediction = restore_oof_prediction(selected_epoch)
test_prediction = inverse_utility_target(test_raw_prediction, final_target_scaler)


def make_prediction_frame(indices, prediction, phase):
    frame = metadata.iloc[indices][[
        'sample_id', 'session', 'symbol', 'decision_timestamp', 'entry_timestamp',
        'target_tp3_3m', 'mfe_3m', 'mae_3m',
    ]].reset_index(drop=True).copy()
    frame['downside_3m'] = downside_3m[indices]
    frame['excursion_utility_lambda1'] = y_utility[indices]
    frame['pred_direct_utility_lambda1'] = prediction
    if phase == 'oof':
        frame['oof_fold'] = split.iloc[indices]['oof_fold'].to_numpy()
    return frame


oof_predictions = make_prediction_frame(oof_indices, oof_prediction, 'oof')
test_predictions = make_prediction_frame(test_indices, test_prediction, 'test')

target_metadata = metadata[[
    'sample_id', 'session', 'symbol', 'decision_timestamp', 'entry_timestamp'
]].copy()
target_metadata['mfe_3m'] = mfe_3m
target_metadata['downside_3m'] = downside_3m
target_metadata['excursion_utility_lambda1'] = y_utility
target_path = PREPROCESS_ROOT / 'ohlc_60m_direct_utility_lambda1_v1_metadata.parquet'
target_schema_path = PREPROCESS_ROOT / 'ohlc_60m_direct_utility_lambda1_v1_schema.json'
target_metadata.to_parquet(target_path, index=False, compression='zstd')
target_schema = {
    'version': 'ohlc_60m_direct_utility_lambda1_v1',
    'base_version': DATA_VERSION,
    'horizon_minutes': 3,
    'lambda_downside': LAMBDA_DOWNSIDE,
    'target_rule': 'mfe_3m - 1.0 * (-mae_3m)',
    'meaning': 'difference of future excursion extremes; not realized return',
    'fold_scaling': 'fit q01/q99 clipping and median/IQR on fit sessions only',
    'changed_from_previous_baseline': 'target and scalar regression loss only',
}
target_schema_path.write_text(
    json.dumps(target_schema, ensure_ascii=False, indent=2), encoding='utf-8'
)

model_path = MODEL_ROOT / 'direct_utility.pt'
checkpoint = {
    'model_class': 'CompactModernTCN',
    'implementation_scope': 'same CompactModernTCN as notebook 06; direct utility target ablation',
    'strategy': 'direct_utility',
    'selected_epoch': selected_epoch,
    'model_config': {
        key: CONFIG[key]
        for key in [
            'input_features', 'context_features', 'd_model', 'd_ff', 'num_blocks',
            'patch_size', 'patch_stride', 'large_kernel_size',
            'small_kernel_size', 'dropout',
        ]
    },
    'output_dim': 1,
    'state_dict': {key: value.detach().cpu() for key, value in final_model.state_dict().items()},
    'input_scaler': {key: np.asarray(value).tolist() for key, value in final_input_scaler.items()},
    'target_scaler': scaler_to_serializable(final_target_scaler),
    'target_definition': target_schema,
    'sequence_features': schema['sequence_features'],
    'context_features': schema['context_features'],
    'data_version': DATA_VERSION,
    'train_sessions': fold_document['train_sessions'],
    'test_sessions': fold_document['test_sessions'],
    'test_is_pristine': False,
    'seed': SEED,
    'parameter_count': MODEL_PARAMETER_COUNT,
}
torch.save(checkpoint, model_path)

artifact_paths = {
    'fold_metrics': RESULT_ROOT / 'oof_daily_metrics.parquet',
    'epoch_selection': RESULT_ROOT / 'oof_epoch_selection.parquet',
    'test_metrics': RESULT_ROOT / 'test_metrics.parquet',
    'training_history': RESULT_ROOT / 'training_history.parquet',
    'oof_predictions': RESULT_ROOT / 'oof_predictions.parquet',
    'test_predictions': RESULT_ROOT / 'test_predictions.parquet',
    'manifest': RESULT_ROOT / 'manifest.json',
}
fold_metrics.to_parquet(artifact_paths['fold_metrics'], index=False, compression='zstd')
epoch_selection.to_parquet(artifact_paths['epoch_selection'], index=False, compression='zstd')
test_metrics.to_parquet(artifact_paths['test_metrics'], index=False, compression='zstd')
training_history.to_parquet(artifact_paths['training_history'], index=False, compression='zstd')
oof_predictions.to_parquet(artifact_paths['oof_predictions'], index=False, compression='zstd')
test_predictions.to_parquet(artifact_paths['test_predictions'], index=False, compression='zstd')

manifest = {
    'experiment': 'moderntcn_direct_utility_lambda1_v1',
    'created_by_notebook': 'notebooks/08_direct_utility_lambda1.ipynb',
    'ablation': 'direct utility target only',
    'baseline_experiment': 'moderntcn_ohlc_60m_v1',
    'target_version': target_schema['version'],
    'device': str(DEVICE),
    'torch_version': torch.__version__,
    'seed': SEED,
    'parameter_count': MODEL_PARAMETER_COUNT,
    'config': CONFIG,
    'selected_epoch': selected_epoch,
    'selection_data': 'Train expanding walk-forward OOF only',
    'test_role': fold_document['test_role'],
    'test_is_pristine': False,
    'model_path': str(model_path),
    'result_paths': {key: str(value) for key, value in artifact_paths.items()},
    'target_paths': {'metadata': str(target_path), 'schema': str(target_schema_path)},
}
artifact_paths['manifest'].write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)

display(pd.DataFrame([
    {'artifact': name, 'path': str(path), 'size_mb': path.stat().st_size / 1024**2}
    for name, path in {
        'model': model_path, **artifact_paths,
        'target_metadata': target_path, 'target_schema': target_schema_path,
    }.items()
]))
print('저장 완료')

,artifact,path,size_mb
0,model,/home/user/urbandatalab/YSLee/model/moderntcn_...,0.479955
1,fold_metrics,/home/user/urbandatalab/YSLee/results/training...,0.010469
2,epoch_selection,/home/user/urbandatalab/YSLee/results/training...,0.006903
3,test_metrics,/home/user/urbandatalab/YSLee/results/training...,0.009019
4,training_history,/home/user/urbandatalab/YSLee/results/training...,0.006820
5,oof_predictions,/home/user/urbandatalab/YSLee/results/training...,0.727843
6,test_predictions,/home/user/urbandatalab/YSLee/results/training...,0.265214
7,manifest,/home/user/urbandatalab/YSLee/results/training...,0.002321
8,target_metadata,/home/user/urbandatalab/YSLee/results/preproce...,1.128310
9,target_schema,/home/user/urbandatalab/YSLee/results/preproce...,0.000401


저장 완료


## 8. 기존 MFE·downside 차감 방식과 직접 비교

기존 baseline의 `pred_mfe_3m - pred_downside_3m`과 새 direct utility를 동일 정답에서 비교한다. Hard classifier 확률도 TP ranking 참고값으로 함께 표시한다.

In [9]:
BASELINE_ROOT = (PROJECT_ROOT / '../../results/training/moderntcn_ohlc_60m_v1').resolve()
baseline_oof = pd.read_parquet(BASELINE_ROOT / 'oof_predictions.parquet')
baseline_test = pd.read_parquet(BASELINE_ROOT / 'test_predictions.parquet')


def score_metrics(frame, score_column):
    actual = frame['excursion_utility_lambda1'].to_numpy()
    score = frame[score_column].to_numpy()
    hard = frame['target_tp3_3m'].to_numpy()
    return {
        'samples': len(frame),
        'utility_spearman': safe_spearman(actual, score),
        'utility_mae': float(mean_absolute_error(actual, score)),
        'hard_tp3_pr_auc': float(average_precision_score(hard, score)),
        'hard_tp3_roc_auc': float(roc_auc_score(hard, score)),
        'score_vs_mfe_spearman': safe_spearman(frame['mfe_3m'], score),
        'score_vs_downside_spearman': safe_spearman(frame['downside_3m'], score),
        'score_mean': float(score.mean()),
    }


comparison_rows = []
comparison_frames = {}
for phase, direct, baseline in [
    ('OOF', oof_predictions, baseline_oof),
    ('Test', test_predictions, baseline_test),
]:
    joined = direct.merge(
        baseline[[
            'sample_id', 'pred_mfe_3m', 'pred_downside_3m',
            'pred_utility_3m', 'pred_open_hard_probability',
        ]],
        on='sample_id', how='left', validate='one_to_one'
    )
    assert joined[['pred_utility_3m', 'pred_open_hard_probability']].notna().all().all()
    comparison_frames[phase] = joined
    for name, column in [
        ('기존 MFE-downside 사후 차감', 'pred_utility_3m'),
        ('새 direct utility', 'pred_direct_utility_lambda1'),
        ('기존 hard classifier 확률', 'pred_open_hard_probability'),
    ]:
        comparison_rows.append({
            'phase': phase, 'model_or_score': name,
            **score_metrics(joined, column),
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison)
print('성공 기준: direct utility Spearman이 기존 차감 방식보다 높고 Test에서도 방향이 유지되어야 합니다.')

,phase,model_or_score,samples,utility_spearman,utility_mae,hard_tp3_pr_auc,hard_tp3_roc_auc,score_vs_mfe_spearman,score_vs_downside_spearman,score_mean
0,OOF,기존 MFE-downside 사후 차감,31081,0.074362,0.018921,0.174022,0.444753,-0.004465,-0.132481,-0.000166
1,OOF,새 direct utility,31081,0.083609,0.018913,0.160862,0.416938,-0.044200,-0.185328,-0.000190
2,OOF,기존 hard classifier 확률,31081,-0.016592,0.152751,0.386128,0.809940,0.476600,0.502898,0.152861
3,Test,기존 MFE-downside 사후 차감,11097,0.096817,0.013857,0.123023,0.438765,0.015560,-0.137093,-0.000356
4,Test,새 direct utility,11097,0.085540,0.013883,0.118091,0.436488,-0.023573,-0.160263,-0.000069
5,Test,기존 hard classifier 확률,11097,-0.026213,0.097687,0.293389,0.803597,0.465175,0.523708,0.098132


성공 기준: direct utility Spearman이 기존 차감 방식보다 높고 Test에서도 방향이 유지되어야 합니다.


## 9. 상위 5% score의 방향성과 downside

예측 score 상위 5%에서 실제 utility, TP 비율과 downside가 개선되는지 확인한다. 이 단계에서는 percentile을 모델 선택이나 진입 threshold로 사용하지 않는다.

In [10]:
top_rows = []
for phase, frame in comparison_frames.items():
    for name, column in [
        ('기존 MFE-downside 사후 차감', 'pred_utility_3m'),
        ('새 direct utility', 'pred_direct_utility_lambda1'),
        ('기존 hard classifier 확률', 'pred_open_hard_probability'),
    ]:
        threshold = frame[column].quantile(0.95)
        selected = frame[frame[column].ge(threshold)]
        top_rows.append({
            'phase': phase,
            'model_or_score': name,
            'selected': len(selected),
            'hard_tp3_precision': float(selected['target_tp3_3m'].mean()),
            'actual_utility_mean': float(selected['excursion_utility_lambda1'].mean()),
            'actual_utility_median': float(selected['excursion_utility_lambda1'].median()),
            'mfe_mean': float(selected['mfe_3m'].mean()),
            'downside_mean': float(selected['downside_3m'].mean()),
            'downside_ge_3pct': float(selected['downside_3m'].ge(0.03).mean()),
        })

top_score_comparison = pd.DataFrame(top_rows)
display(top_score_comparison)
print('다음 단계는 이 결과를 확정한 뒤 연속 positive를 episode로 묶어 첫 진입 기준으로 재평가합니다.')

,phase,model_or_score,selected,hard_tp3_precision,actual_utility_mean,actual_utility_median,mfe_mean,downside_mean,downside_ge_3pct
0,OOF,기존 MFE-downside 사후 차감,1555,0.297106,0.001762,0.004785,0.025698,0.023936,0.265595
1,OOF,새 direct utility,1555,0.270740,0.002737,0.005291,0.023791,0.021054,0.222508
2,OOF,기존 hard classifier 확률,1555,0.515113,-0.001243,-0.002435,0.042254,0.043496,0.535691
3,Test,기존 MFE-downside 사후 차감,555,0.219820,0.003095,0.005602,0.018893,0.015798,0.162162
4,Test,새 direct utility,558,0.184588,0.002687,0.005407,0.017604,0.014917,0.145161
5,Test,기존 hard classifier 확률,556,0.377698,-0.002549,-0.004183,0.030373,0.032922,0.465827


다음 단계는 이 결과를 확정한 뒤 연속 positive를 episode로 묶어 첫 진입 기준으로 재평가합니다.
